In [ ]:
import math as M
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define the Robot Configuration

In [ ]:
l1 = 10
l2 = 7

# Generate the Dataset (Forward Kinematics): 50,000 sample

In [ ]:
def forward_kinematics(l1, l2, theta1, theta2):
  theta1_rad = M.radians(theta1)
  theta2_rad = M.radians(theta2)
  x1 = l1 * M.cos(theta1_rad)
  x2 = x1 + l2 * M.cos(theta1_rad + theta2_rad)
  y1 = l1 * M.sin(theta1_rad)
  y2 = y1 + l2 * M.sin(theta1_rad + theta2_rad)
  return (x2, y2)

In [ ]:
dataset = np.zeros((50000, 4))
for i in range(50000):
  theta1 = np.random.uniform(-180, 180)
  theta2 = np.random.uniform(-180, 180)
  x2, y2 = forward_kinematics(l1, l2, theta1, theta2)
  dataset[i, 0] = theta1
  dataset[i, 1] = theta2
  dataset[i, 2] = x2
  dataset[i, 3] = y2

dataset = dataset.astype(np.float32)

# Filter the data

In [ ]:
elbow_down_mask = dataset[:, 1] >= 0

dataset_elbow_down = dataset[elbow_down_mask]

print(f"Total: {len(dataset)}")
print(f"elbow-down: {len(dataset_elbow_down)}")

Total: 50000
elbow-down: 25110


# Preprocess the Data (split, normalization)

In [ ]:
X_down = dataset_elbow_down[:, 2:4]
Y_down = dataset_elbow_down[:, 0:2]
X_train_down, X_test_down, Y_train_down, Y_test_down = train_test_split(
    X_down, Y_down, test_size=0.2, random_state=42
)

scaler_down_X = StandardScaler()
X_train_down_scaled = scaler_down_X.fit_transform(X_train_down)
X_test_down_scaled = scaler_down_X.transform(X_test_down)


print(f"Train Features: {X_train_down_scaled.shape}, Train Targets: {Y_train_down.shape}")
print(f"Test Features:  {X_test_down_scaled.shape}, Test Targets:  {Y_test_down.shape}")

Train Features: (20088, 2), Train Targets: (20088, 2)
Test Features:  (5022, 2), Test Targets:  (5022, 2)


# Build the ANN

In [ ]:
nn = keras.Sequential()
nn.add(layers.Dense(64, activation="relu", input_shape=(2,)))
nn.add(layers.Dense(32, activation="relu"))
nn.add(layers.Dense(16, activation="relu"))
nn.add(layers.Dense(2))
nn.compile(optimizer="adam", loss="mean_squared_error", metrics=["mae"])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# Train the Model

In [ ]:
model = nn.fit(
    X_train_down_scaled,
    Y_train_down,
    epochs=100,
    batch_size=32,
    validation_data=(X_test_down_scaled, Y_test_down))

Epoch 1/100
628/628 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 5193.4175 - mae: 56.1016 - val_loss: 2900.8958 - val_mae: 40.1507
Epoch 2/100
628/628 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 2329.2683 - mae: 32.4758 - val_loss: 1670.3413 - val_mae: 23.4456
Epoch 3/100
628/628 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 1568.3776 - mae: 20.0523 - val_loss: 1341.8566 - val_mae: 17.2993
Epoch 4/100
628/628 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1353.2625 - mae: 16.7825 - val_loss: 1198.8055 - val_mae: 15.2305
Epoch 5/100
628/628 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1230.3002 - mae: 15.0513 - val_loss: 1110.1542 - val_mae: 14.2457
Epoch 6/100
628/628 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1144.8944 - mae: 14.2151 - val_loss: 1029.1199 - val_mae: 13.2361
Epoch 7/100
628/628 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1064.3540 - mae: 13.6781 - val_loss: 964.7324 - val_mae: 13.1833
Epoch 8/100
628/628 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 991.7360 - mae: 13.0895 - val_loss: 881.7276 - val_

# Test the model

In [ ]:
test_cases = np.array([[10, 7], [8.6603, 12]])
test_cases_scaled = scaler_down_X.transform(test_cases)
predicted_angles = nn.predict(test_cases_scaled)
predicted_angles

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step


array([[-0.7226948, 90.97722  ],
       [26.184769 , 66.10464  ]], dtype=float32)